# 축3 자가소비 적합성 분석
## 02. 전처리

### 개요
- 축2 전처리 데이터(742개) 기준으로 축3 필요 변수 추가
- 추가 변수: 운영시간, 자치구, EV 충전소 반경 500m 집계, 자치구별 EV 등록대수

### 데이터 출처
| 변수 | 출처 |
|------|------|
| 운영시간 | OA-13122 원본 CSV |
| 자치구 | 주소에서 파싱 |
| EV 충전소 500m | 한국환경공단 API (서울_EV충전소.csv) |
| EV 등록대수 | 서울시 OA-15640 |

In [1]:
import pandas as pd
import numpy as np

# 축2 전처리 데이터 로드
df = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\parking_preprocessed.csv',
    encoding='utf-8-sig'
)

# OA-13122 원본 로드
parking_raw = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\raw\서울시 공영주차장 안내 정보(원본).csv',
    encoding='cp949'
)

print(f"축2 전처리 데이터: {df.shape}")
print(f"원본 데이터: {parking_raw.shape}")
print(f"\n원본 컬럼: {parking_raw.columns.tolist()}")

축2 전처리 데이터: (742, 9)
원본 데이터: (3696, 39)

원본 컬럼: ['주차장코드', '주차장명', '주소', '주차장 종류', '주차장 종류명', '운영구분', '운영구분명', '전화번호', '주차현황 정보 제공여부', '주차현황 정보 제공여부명', '총 주차면', '유무료구분', '유무료구분명', '야간무료개방여부', '야간무료개방여부명', '평일 운영 시작시각(HHMM)', '평일 운영 종료시각(HHMM)', '주말 운영 시작시각(HHMM)', '주말 운영 종료시각(HHMM)', '공휴일 운영 시작시각(HHMM)', '공휴일 운영 종료시각(HHMM)', '최종데이터 동기화 시간', '토요일 유,무료 구분', '토요일 유,무료 구분명', '공휴일 유,무료 구분', '공휴일 유,무료 구분명', '월 정기권 금액', '노상 주차장 관리그룹번호', '기본 주차 요금', '기본 주차 시간(분 단위)', '추가 단위 요금', '추가 단위 시간(분 단위)', '버스 기본 주차 요금', '버스 기본 주차 시간(분 단위)', '버스 추가 단위 시간(분 단위)', '버스 추가 단위 요금', '일 최대 요금', '위도', '경도']


### Step 1. 운영시간 추출
- OA-13122 원본에서 평일/주말 운영시간 추출
- HHMM 정수형 → 시간 단위 변환
- 2400(자정) 예외처리
- 주차장명 기준 중복 제거 후 축2 데이터에 조인

In [2]:
def calc_hours(start, end):
    try:
        s = int(str(int(start)).zfill(4)[:2])
        e = int(str(int(end)).zfill(4)[:2])
        if e == 24:
            e = 24
        hours = e - s
        return hours if hours > 0 else 24
    except:
        return None

# 운영시간 계산
parking_raw['평일운영시간'] = parking_raw.apply(
    lambda r: calc_hours(r['평일 운영 시작시각(HHMM)'], r['평일 운영 종료시각(HHMM)']), axis=1)
parking_raw['주말운영시간'] = parking_raw.apply(
    lambda r: calc_hours(r['주말 운영 시작시각(HHMM)'], r['주말 운영 종료시각(HHMM)']), axis=1)

# 주차장명 기준 중복 제거 (운영시간 중앙값)
ops_time = parking_raw.groupby('주차장명')[['평일운영시간', '주말운영시간']].median().reset_index()

print(f"운영시간 추출 완료: {ops_time.shape}")
print(ops_time[['주차장명', '평일운영시간', '주말운영시간']].head(10).to_string())
print(f"\n결측: 평일 {ops_time['평일운영시간'].isna().sum()}개, 주말 {ops_time['주말운영시간'].isna().sum()}개")

운영시간 추출 완료: (1694, 3)
               주차장명  평일운영시간  주말운영시간
0  (구)동작구청 부설주차장(구)    24.0    24.0
1      (구)북부지법주변(구)    10.0     9.0
2      01-05-023(구)    24.0    24.0
3      01-09-102(구)    24.0    24.0
4      01-09-106(구)    24.0    24.0
5      01-09-113(구)    24.0    24.0
6      01-26-036(구)    24.0    24.0
7      01-26-049(구)    24.0    24.0
8      01-26-052(구)    24.0    24.0
9      01-37-114(구)    24.0    24.0

결측: 평일 0개, 주말 3개


### Step 2. 축2 데이터와 운영시간 조인
- pklt_nm(축2) ↔ 주차장명(원본) 매칭 확인

In [3]:
# 매칭 확인
ax2_names = set(df['pklt_nm'].tolist())
raw_names = set(ops_time['주차장명'].tolist())

matched = ax2_names & raw_names
unmatched = ax2_names - raw_names

print(f"축2 주차장 수: {len(ax2_names)}")
print(f"매칭 성공: {len(matched)}개")
print(f"매칭 실패: {len(unmatched)}개")
print(f"\n매칭 실패 샘플 (10개):")
for name in list(unmatched)[:10]:
    print(f"  {name}")

축2 주차장 수: 741
매칭 성공: 0개
매칭 실패: 741개

매칭 실패 샘플 (10개):
  구로4동 마을공동 주차장
  새마을시장
  면목유수지 공영주차장
  배봉공영주차장
  반포2동 공영주차장
  주공5단지옆
  둘리뮤지엄 공영주차장
  광장공영주차장
  영등포로터리
  동화동 공영주차장


### Step 2-1. 주차장명 정규화 후 매칭
- 축2 전처리 시 적용한 정규화 규칙 동일하게 원본에 적용
- (구), (시), 괄호 설명, 공백 제거 후 매칭

In [4]:
import re

def normalize_name(name):
    if pd.isna(name):
        return name
    name = str(name)
    name = re.sub(r'\(시\)|\(구\)|舊', '', name)  # (시), (구), 舊 제거
    name = re.sub(r'\([^)]*\)', '', name)          # 괄호 안 내용 제거
    name = re.sub(r'\s+', '', name)                # 공백 제거
    return name.strip()

# 양쪽 정규화 키 생성
df['nm_key'] = df['pklt_nm'].apply(normalize_name)
ops_time['nm_key'] = ops_time['주차장명'].apply(normalize_name)

# 정규화 후 매칭 확인
ax2_keys = set(df['nm_key'].tolist())
raw_keys = set(ops_time['nm_key'].tolist())

matched = ax2_keys & raw_keys
unmatched = ax2_keys - raw_keys

print(f"정규화 후 매칭 성공: {len(matched)}개")
print(f"정규화 후 매칭 실패: {len(unmatched)}개")
print(f"\n매칭 실패 샘플 (10개):")
for name in list(unmatched)[:10]:
    print(f"  {name}")

정규화 후 매칭 성공: 727개
정규화 후 매칭 실패: 0개

매칭 실패 샘플 (10개):


### Step 2-2. 매칭 누락 확인
- 742개 중 727개 매칭 → 15개 누락 원인 파악

In [5]:
# 15개 누락 확인
missing_keys = ax2_keys - raw_keys
print(f"누락된 키: {len(missing_keys)}개")

# nm_key 기준으로 조인 후 결측 확인
df_merged = df.merge(
    ops_time[['nm_key', '평일운영시간', '주말운영시간']],
    on='nm_key',
    how='left'
)

missing_rows = df_merged[df_merged['평일운영시간'].isna()]
print(f"\n운영시간 결측 주차장 ({len(missing_rows)}개):")
print(missing_rows[['pklt_nm', 'nm_key']].to_string())

누락된 키: 0개

운영시간 결측 주차장 (0개):
Empty DataFrame
Columns: [pklt_nm, nm_key]
Index: []


- 결측 0개, 742개 다 매칭됨.
- 아까 727개는 중복 키 때문에 착시 발생(여러 행이 같은 nm_key를 가진 경우).

### Step 2-3. 운영시간 조인 완료
- 742개 전체 매칭 성공
- 평일/주말 운영시간 평균으로 단일화

In [6]:
# 운영시간 평균 단일화
df_merged['운영시간'] = (df_merged['평일운영시간'] + df_merged['주말운영시간']) / 2

print(f"shape: {df_merged.shape}")
print(f"결측 확인:")
print(df_merged[['평일운영시간', '주말운영시간', '운영시간']].isna().sum())
print(f"\n운영시간 기초통계:")
print(df_merged['운영시간'].describe())

shape: (779, 13)
결측 확인:
평일운영시간    0
주말운영시간    4
운영시간      4
dtype: int64

운영시간 기초통계:
count    775.000000
mean      18.073226
std        6.736208
min        6.500000
25%       10.000000
50%       24.000000
75%       24.000000
max       24.000000
Name: 운영시간, dtype: float64


### Step 2-4. 중복 조인 문제 해결
- nm_key 중복으로 742 → 779행 발생
- ops_time에서 nm_key 기준 중복 제거 후 재조인

In [7]:
# 중복 nm_key 확인
dup_keys = ops_time[ops_time['nm_key'].duplicated(keep=False)]
print(f"ops_time 중복 nm_key: {ops_time['nm_key'].duplicated().sum()}개")
print(dup_keys[['주차장명', 'nm_key', '평일운영시간', '주말운영시간']].head(20).to_string())

ops_time 중복 nm_key: 80개
                         주차장명       nm_key  평일운영시간  주말운영시간
283            공유) 10-05구역(구)   공유)10-05구역     9.0    24.0
284        공유) 10-05구역(노외)(구)   공유)10-05구역     9.0    24.0
307            공유) 11-02구역(구)   공유)11-02구역     9.0    24.0
308        공유) 11-02구역(노외)(구)   공유)11-02구역     9.0    24.0
345            공유) 12-09구역(구)   공유)12-09구역     9.0    24.0
346        공유) 12-09구역(노외)(구)   공유)12-09구역     9.0    24.0
458  공유) 광희동 01(001~007구간)(구)     공유)광희동01    23.0    23.0
459  공유) 광희동 01(014~019구간)(구)     공유)광희동01    23.0    23.0
476     공유) 동작도서관(01~05구간)(구)     공유)동작도서관     9.0    24.0
477     공유) 동작도서관(06~10구간)(구)     공유)동작도서관     9.0    24.0
478     공유) 동작도서관(11~15구간)(구)     공유)동작도서관     9.0    24.0
479     공유) 동작도서관(16~19구간)(구)     공유)동작도서관     9.0    24.0
480     공유) 동작도서관(22~26구간)(구)     공유)동작도서관     9.0    24.0
481     공유) 동작도서관(67~70구간)(구)     공유)동작도서관     9.0    24.0
482     공유) 동작도서관(71~76구간)(구)     공유)동작도서관     9.0    24.0
485   공유) 면목2동 10구간(25~27)(구)  공

### Step 2-5. nm_key 중복 제거 후 재조인
- 동일 nm_key 내 운영시간이 동일하므로 첫 번째 행 사용

In [8]:
# nm_key 중복 제거
ops_time_dedup = ops_time.drop_duplicates(subset='nm_key', keep='first')
print(f"중복 제거 후 ops_time: {len(ops_time_dedup)}개")

# 재조인
df_merged = df.merge(
    ops_time_dedup[['nm_key', '평일운영시간', '주말운영시간']],
    on='nm_key',
    how='left'
)

# 운영시간 평균 단일화
df_merged['운영시간'] = (df_merged['평일운영시간'] + df_merged['주말운영시간']) / 2

print(f"조인 후 shape: {df_merged.shape}")
print(f"\n결측 확인:")
print(df_merged[['평일운영시간', '주말운영시간', '운영시간']].isna().sum())
print(f"\n운영시간 기초통계:")
print(df_merged['운영시간'].describe())

중복 제거 후 ops_time: 1614개
조인 후 shape: (742, 13)

결측 확인:
평일운영시간    0
주말운영시간    2
운영시간      2
dtype: int64

운영시간 기초통계:
count    740.000000
mean      18.056419
std        6.745139
min        6.500000
25%       10.000000
50%       24.000000
75%       24.000000
max       24.000000
Name: 운영시간, dtype: float64


### Step 2-6. 주말운영시간 결측 2개 확인

In [9]:
missing = df_merged[df_merged['주말운영시간'].isna()]
print(missing[['pklt_nm', 'nm_key', '평일운영시간', '주말운영시간']].to_string())

   pklt_nm  nm_key  평일운영시간  주말운영시간
96  구룡산제1호  구룡산제1호    10.0     NaN
97  구룡산제2호  구룡산제2호    10.0     NaN


### Step 2-7. 결측 2개 원본 확인 및 처리

In [10]:
# 원본에서 확인
print(parking_raw[parking_raw['주차장명'].str.contains('구룡산', na=False)][
    ['주차장명', '평일 운영 시작시각(HHMM)', '평일 운영 종료시각(HHMM)',
     '주말 운영 시작시각(HHMM)', '주말 운영 종료시각(HHMM)']
].to_string())

           주차장명  평일 운영 시작시각(HHMM)  평일 운영 종료시각(HHMM)  주말 운영 시작시각(HHMM)  주말 운영 종료시각(HHMM)
2669  구룡산제2호(구)             900.0            1900.0               NaN               NaN
2678  구룡산제1호(구)             900.0            1900.0               NaN               NaN


### Step 2-7. 결측 처리
- 구룡산제1·2호: 원본 자체에 주말 운영시각 없음
- 평일만 운영하는 주차장으로 판단 → 주말운영시간 = 0으로 처리
- 운영시간 = (평일 + 0) / 2

In [11]:
# 주말운영시간 결측 → 0으로 처리
df_merged['주말운영시간'] = df_merged['주말운영시간'].fillna(0)
df_merged['운영시간'] = (df_merged['평일운영시간'] + df_merged['주말운영시간']) / 2

print(f"결측 확인:")
print(df_merged[['평일운영시간', '주말운영시간', '운영시간']].isna().sum())
print(f"\n구룡산 확인:")
print(df_merged[df_merged['pklt_nm'].str.contains('구룡산', na=False)][
    ['pklt_nm', '평일운영시간', '주말운영시간', '운영시간']
].to_string())

결측 확인:
평일운영시간    0
주말운영시간    0
운영시간      0
dtype: int64

구룡산 확인:
   pklt_nm  평일운영시간  주말운영시간  운영시간
96  구룡산제1호    10.0     0.0   5.0
97  구룡산제2호    10.0     0.0   5.0


### Step 2-8. 운영시간 가중평균 적용
- 평일 5일, 주말 2일 기준 가중평균
- 운영시간 = (평일운영시간 × 5 + 주말운영시간 × 2) / 7
- 주말 미운영(구룡산제1·2호)은 주말운영시간 = 0으로 처리

In [12]:
# 가중평균 운영시간
df_merged['운영시간'] = (df_merged['평일운영시간'] * 5 + df_merged['주말운영시간'] * 2) / 7

print(f"결측 확인: {df_merged['운영시간'].isna().sum()}")
print(f"\n운영시간 기초통계:")
print(df_merged['운영시간'].describe())
print(f"\n구룡산 확인:")
print(df_merged[df_merged['pklt_nm'].str.contains('구룡산', na=False)][
    ['pklt_nm', '평일운영시간', '주말운영시간', '운영시간']
].to_string())

결측 확인: 0

운영시간 기초통계:
count    742.000000
mean      17.896900
std        6.638483
min        7.142857
25%       10.000000
50%       24.000000
75%       24.000000
max       24.000000
Name: 운영시간, dtype: float64

구룡산 확인:
   pklt_nm  평일운영시간  주말운영시간      운영시간
96  구룡산제1호    10.0     0.0  7.142857
97  구룡산제2호    10.0     0.0  7.142857


### 운영시간 전처리 결과
- 742개 전체 결측 없음
- 가중평균 운영시간 범위: 7.1 ~ 24.0시간
- 중앙값 24.0시간 (24시간 운영 주차장이 많음)
- 구룡산제1·2호: 주말 미운영 → 가중평균 7.14시간

### Step 3. 자치구 추출
- 주소(addr)에서 자치구 파싱

In [13]:
# 자치구 추출
df_merged['자치구'] = df_merged['addr'].str.extract(r'(\S+구)')

print(f"자치구 추출 결측: {df_merged['자치구'].isna().sum()}개")
print(f"\n자치구 분포:")
print(df_merged['자치구'].value_counts())

자치구 추출 결측: 0개

자치구 분포:
자치구
영등포구    75
양천구     64
강남구     59
강서구     43
송파구     40
금천구     36
서초구     35
중구      32
종로구     29
마포구     28
구로구     26
강북구     26
노원구     25
용산구     25
강동구     24
성동구     23
동대문구    22
동작구     22
도봉구     20
관악구     20
은평구     16
중랑구     15
성북구     13
광진구     13
서대문구    11
Name: count, dtype: int64


### 자치구 추출 결과
- 742개 전체 결측 없음
- 영등포구(75개), 양천구(64개), 강남구(59개) 순으로 많음
- EDA 결과와 일치

### Step 4. EV 충전소 반경 500m 집계
- 주차장 좌표 기준 반경 500m 내 EV 충전소 수 집계
- Haversine 공식 사용
- 데이터: 서울_EV충전소.csv (74,024개)

In [14]:
from math import radians, sin, cos, sqrt, atan2

# EV 충전소 데이터 로드
df_ev = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\서울_EV충전소.csv',
    encoding='utf-8-sig'
)

df_ev['lat'] = pd.to_numeric(df_ev['lat'], errors='coerce')
df_ev['lng'] = pd.to_numeric(df_ev['lng'], errors='coerce')
df_ev = df_ev.dropna(subset=['lat', 'lng'])

print(f"EV 충전소 수: {len(df_ev)}개")

# Haversine 함수
def haversine(lat1, lng1, lat2, lng2):
    R = 6371000
    lat1, lng1, lat2, lng2 = map(radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlng/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# 반경 500m 내 EV 충전소 수 집계
ev_lat = df_ev['lat'].values
ev_lng = df_ev['lng'].values

results = []
for _, row in df_merged.iterrows():
    count = sum(
        haversine(row['lat'], row['lot'], lat, lng) <= 500
        for lat, lng in zip(ev_lat, ev_lng)
    )
    results.append(count)

df_merged['ev_충전소_500m'] = results
print(f"\n집계 완료")
print(df_merged[['pklt_nm', 'ev_충전소_500m']].sort_values('ev_충전소_500m', ascending=False).head(10).to_string())

EV 충전소 수: 74024개

집계 완료
            pklt_nm  ev_충전소_500m
680  하왕십리(건물) 공영주차장          464
219           동우주차장          464
126   남가좌2동 제1공영주차장          450
127   남가좌2동 제2공영주차장          450
182      대현산배수지 주차장          442
557            우리은행          416
560    웅진코웨이~sk트윈타워          416
462              아톰          416
115            기륭전자          416
50         견인보관소 노외          416


### EV 충전소 반경 500m 집계 결과
- 742개 전체 집계 완료
- 최다: 하왕십리 공영주차장, 동우주차장 (464개)
- 이전 100개 기준 분석 대비 상위 주차장 변동 있음 (모수 확대 영향)

### Step 5. 자치구별 EV 등록대수 조인
- 서울시 자치구별 연료별 자동차 등록현황 (OA-15640, 2026년 3월 기준)
- 전기차만 필터링 후 자치구별 합산

In [15]:
# EV 등록대수 데이터 로드
df_car = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\raw\서울시 자치구별 연료별 차종별 용도별 등록현황(26년3월).csv',
    encoding='cp949'
)

# 전기차만 필터링
ev_by_gu = df_car[
    (df_car['연료명'] == '전기') &
    (df_car['시군구명'] != '합 계')
].groupby('시군구명')['계'].sum().reset_index()
ev_by_gu.columns = ['자치구', 'ev_등록대수']

# 조인
df_merged = df_merged.merge(ev_by_gu, on='자치구', how='left')

print(f"결측: {df_merged['ev_등록대수'].isna().sum()}개")
print(f"shape: {df_merged.shape}")
print(df_merged[['pklt_nm', '자치구', 'ev_등록대수']].head(10).to_string())

결측: 0개
shape: (742, 16)
         pklt_nm   자치구  ev_등록대수
0      15-가600구간   강동구     5653
1          21-37   송파구     7851
2            BYC   금천구     2257
3   KBS별관뒤 노상주차장  영등포구     4323
4   KBS별관옆 노상주차장  영등포구     4323
5  KBS본관 앞 노상주차장  영등포구     4323
6         KBS본관뒤  영등포구     4323
7          LG텔레콤   금천구     2257
8  MBC방송국뒤 노상주차장  영등포구     4323
9         SJ테크노빌   금천구     2257


### EV 등록대수 조인 결과
- 742개 전체 결측 없음
- 자치구별 EV 등록대수 정상 조인

### Step 6. 전처리 완료 데이터 저장
- 축3 분석에 필요한 컬럼만 선택
- 중간 계산용 컬럼(nm_key 등) 제거

In [16]:
# 필요 컬럼만 선택
cols = [
    'pklt_nm',        # 주차장명
    'pklt_knd_nm',    # 노상/노외
    'lat', 'lot',     # 위경도
    'addr',           # 주소
    '자치구',          # 자치구
    '평일운영시간',     # 평일 운영시간
    '주말운영시간',     # 주말 운영시간
    '운영시간',        # 가중평균 운영시간
    'ev_충전소_500m',  # 반경 500m EV 충전소 수
    'ev_등록대수',     # 자치구별 EV 등록대수
]

df_axis3 = df_merged[cols].copy()

print(f"최종 shape: {df_axis3.shape}")
print(f"결측 확인:")
print(df_axis3.isna().sum())

# 저장
df_axis3.to_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\parking_axis3_preprocessed.csv',
    index=False,
    encoding='utf-8-sig'
)
print("\n저장 완료")

최종 shape: (742, 11)
결측 확인:
pklt_nm        0
pklt_knd_nm    0
lat            0
lot            0
addr           0
자치구            0
평일운영시간         0
주말운영시간         0
운영시간           0
ev_충전소_500m    0
ev_등록대수        0
dtype: int64

저장 완료


### 전처리 최종 결과
- 742개 전체 결측 없음
- 저장: `parking_axis3_preprocessed.csv`

| 컬럼 | 설명 |
|------|------|
| pklt_nm | 주차장명 |
| pklt_knd_nm | 노상/노외 구분 |
| lat, lot | 위경도 |
| addr | 주소 |
| 자치구 | 주소에서 파싱 |
| 평일운영시간 | 평일 운영시간 (시간) |
| 주말운영시간 | 주말 운영시간 (시간) |
| 운영시간 | 가중평균 운영시간 (평일×5 + 주말×2) / 7 |
| ev_충전소_500m | 반경 500m 내 EV 충전소 수 |
| ev_등록대수 | 자치구별 전기차 등록대수 (2026년 3월) |